In [1]:
import json

import fitz
import json
import requests
import io
import pickle
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import re
from bs4 import BeautifulSoup
import pandas as pd
import nltk
import random

In [2]:
# find pdf folder
source_dir = "/srv/data/tome/tome-corpus/EMLAP_2025-10-31/pdfs_only/"
len(os.listdir(source_dir))

100

In [3]:
# for each PDF file:
# for each page in the PDF file
# collect info about its height and width
# store it

source_dir = "/srv/data/tome/tome-corpus/EMLAP_2025-10-31/pdfs_only/"
docs_pagesizes = {}
for filename in os.listdir(source_dir):
    try:
        #filename = [f for f in os.listdir(os.path.join(source_dir, dir)) if ".pdf" in f][0]
        filepath = os.path.join(source_dir, filename)
        doc = fitz.open(filepath)
        doc_pagesizes = []
        for n, p in enumerate(doc):
            pix = p.get_pixmap()
            page_data = {"page_index" : n, "page_width" : pix.width, "page_height" : pix.height}
            doc_pagesizes.append(page_data)
        docs_pagesizes[filename[:6]] = doc_pagesizes
    except:
        pass

In [9]:
with open("../data/docs_pagesizes.json", 'w') as f:
    json.dump(docs_pagesizes, f, indent=2, ensure_ascii=False)

In [ ]:
# we will make the recalculations in paralel both for:
# (1) raw annotated textblocks (textblocks with text format as extracted from the PDF, but enriched by the "tag" attribute
# (2) annotated textblocks with sanitized text (=normalized, clean text, with autocompleted tag pairs etc)

In [36]:
path_sanizited_textblocks = "../data/emlap_sanitized_textblocks/"
textblocks_sanitized_filenames = os.listdir(path_sanizited_textblocks)
textblocks_sanitized_filenames[:10]

['100084_Croll1609_Basilica_chymica_MDZ_MBS.json',
 '100094_Anon1625_Musaeum_hermeticum_VD17_SLUB.json',
 '100058_Hagecius1596_Actio_medica_ER_UBB.json',
 '100013_Ulstad1525_Coelum_philosophorum_Medica_BIUSP.json',
 '100069_Severinus1571_Idea_medicinae_philosophicae_GB_Noscemus.json',
 '100074_Hoghelande1595_De_lapidis_physici_conditionibus_MDZ_MBS.json',
 '100059_Suavius1567_Theophrasti_Paracelsi_Philosophiae_ONB.json',
 '100053_Mirandola1586_De_auro_libri_tres_MDZ_MBS.json',
 '100060_Witestein1583_Disceptatio_philosophica_MDZ_MBS.json',
 '100071_Pseudo-Lull1566_Testamentum_MDZ_MBS.json']

In [32]:
path_annotated_textblocks = "../data/emlap_annotated_textblocks/"
textblocks_annotated_filenames = os.listdir(path_annotated_textblocks)
textblocks_annotated_filenames = [fn for fn in textblocks_annotated_filenames if "_params.json" not in fn]
textblocks_annotated_filenames[:10]

['100084_Croll1609_Basilica_chymica_MDZ_MBS.json',
 '100094_Anon1625_Musaeum_hermeticum_VD17_SLUB.json',
 '100058_Hagecius1596_Actio_medica_ER_UBB.json',
 '100013_Ulstad1525_Coelum_philosophorum_Medica_BIUSP.json',
 '100069_Severinus1571_Idea_medicinae_philosophicae_GB_Noscemus.json',
 '100074_Hoghelande1595_De_lapidis_physici_conditionibus_MDZ_MBS.json',
 '100059_Suavius1567_Theophrasti_Paracelsi_Philosophiae_ONB.json',
 '100053_Mirandola1586_De_auro_libri_tres_MDZ_MBS.json',
 '100060_Witestein1583_Disceptatio_philosophica_MDZ_MBS.json',
 '100071_Pseudo-Lull1566_Testamentum_MDZ_MBS.json']

In [33]:
# test with individual document
filename = textblocks_sanitized_filenames[0]
# load textblocks
with open(os.path.join(path_sanizited_textblocks, filename), 'r', ) as f:
    doc_sanitized_textblocks_json = json.load(f)

with open(os.path.join(path_annotated_textblocks, filename), 'r', ) as f:
    doc_annotated_textblocks_json = json.load(f)
# map on it corresponding previously extracted page sizes
doc_pagesizes = docs_pagesizes[filename[:6]]

In [20]:
doc_pagesizes[:10]

[{'page_index': 0, 'page_width': 595, 'page_height': 842},
 {'page_index': 1, 'page_width': 2120, 'page_height': 2688},
 {'page_index': 2, 'page_width': 1896, 'page_height': 2560},
 {'page_index': 3, 'page_width': 1984, 'page_height': 2528},
 {'page_index': 4, 'page_width': 1936, 'page_height': 2528},
 {'page_index': 5, 'page_width': 2120, 'page_height': 2528},
 {'page_index': 6, 'page_width': 1896, 'page_height': 2528},
 {'page_index': 7, 'page_width': 2112, 'page_height': 2528},
 {'page_index': 8, 'page_width': 1904, 'page_height': 2544},
 {'page_index': 9, 'page_width': 2112, 'page_height': 2528}]

In [22]:
def recalculation(doc_textblocks_json, doc_pagesizes):
    doc_textblocks_recalculated = []
    for  page_textblocks, page_size in zip(doc_textblocks_json, doc_pagesizes):
        page_width = page_size["page_width"]
        page_height = page_size["page_height"]
        page_textblocks_recalculated = []
        for textblock_data in page_textblocks:
                        textblock_data_recalculated = {
                            "coordinates":
                                {"upper_left_x": textblock_data["coordinates"][0] / page_width,
                                 "upper_left_y": textblock_data["coordinates"][1] / page_height,
                                 "lower_right_x" : textblock_data["coordinates"][2] / page_width,
                                 "lower_right_y": textblock_data["coordinates"][3] / page_height},
                            "text": textblock_data["text"],
                            "tag": textblock_data["tag"]}
                        page_textblocks_recalculated.append(textblock_data_recalculated)
        doc_textblocks_recalculated.append(page_textblocks_recalculated)
    return doc_textblocks_recalculated

In [26]:
doc_annotated_textblocks_recalculated = recalculation(doc_annotated_textblocks_json, doc_pagesizes)
doc_sanitized_textblocks_recalculated = recalculation(doc_sanitized_textblocks_json, doc_pagesizes)

In [25]:
doc_annotated_textblocks_recalculated[10][:5]

[{'coordinates': {'upper_left_x': 0.1805084681106826,
   'upper_left_y': 0.06823348999023438,
   'lower_right_x': 0.9509123462741658,
   'lower_right_y': 0.07012623567310418},
  'text': 'fuerit, cum non solum huiusmodi libris perperam quippe recusis & adductis (quos\n',
  'tag': 'text'},
 {'coordinates': {'upper_left_x': 0.17580509185791016,
   'upper_left_y': 0.09019880565559074,
   'lower_right_x': 0.9517700066000728,
   'lower_right_y': 0.09213886802504867},
  'text': 'quidem antedictus Osualdus Crollius illiusque haeredes & mandatarii vbicunque\n',
  'tag': 'text'},
 {'coordinates': {'upper_left_x': 0.1766949184870316,
   'upper_left_y': 0.1122492480353226,
   'lower_right_x': 0.955103146827827,
   'lower_right_y': 0.1141893164216532},
  'text': 'depraehensos siue propriâ authoritate vel Magistratus loci illius auxilio sibi ven¬\n',
  'tag': 'text'},
 {'coordinates': {'upper_left_x': 0.17478813559322035,
   'upper_left_y': 0.1351419803848026,
   'lower_right_x': 0.9481065394514698,

In [27]:
doc_sanitized_textblocks_recalculated[10][:5]


[{'coordinates': {'upper_left_x': 0.1805084681106826,
   'upper_left_y': 0.06823348999023438,
   'lower_right_x': 0.9509123462741658,
   'lower_right_y': 0.07012623567310418},
  'text': 'fuerit, cum non solum huiusmodi libris perperam quippe recusis & adductis (quos ',
  'tag': 'text'},
 {'coordinates': {'upper_left_x': 0.17580509185791016,
   'upper_left_y': 0.09019880565559074,
   'lower_right_x': 0.9517700066000728,
   'lower_right_y': 0.09213886802504867},
  'text': 'quidem antedictus Osualdus Crollius illiusque haeredes & mandatarii ubicunque ',
  'tag': 'text'},
 {'coordinates': {'upper_left_x': 0.1766949184870316,
   'upper_left_y': 0.1122492480353226,
   'lower_right_x': 0.955103146827827,
   'lower_right_y': 0.1141893164216532},
  'text': 'depraehensos siue propria authoritate uel Magistratus loci illius auxilio sibi uen',
  'tag': 'text'},
 {'coordinates': {'upper_left_x': 0.17478813559322035,
   'upper_left_y': 0.1351419803848026,
   'lower_right_x': 0.9481065394514698,
   '

In [28]:
outdir_recalculated_sanitized_textblocks = "../data/emlap_recalculated_sanitized_textblocks/"
os.makedirs(outdir_recalculated_sanitized_textblocks, exist_ok=True)
outdir_recalculated_annotated_textblocks = "../data/emlap_recalculated_annotated_textblocks/"
os.makedirs(outdir_recalculated_annotated_textblocks, exist_ok=True)

In [30]:
path_annotated_textblocks

'../data/emlap_annotated_textblocks'

In [ ]:
path_annotated_textblocks

In [35]:
# first recalculate with raw annotated
for filename in textblocks_annotated_filenames:
        with open(os.path.join(path_annotated_textblocks, filename), 'r', ) as f:
            doc_textblocks_json = json.load(f)
        try:
            print("working on: ", filename, " ...")
            doc_pagesizes = docs_pagesizes[filename[:6]]
            doc_textblocks_recalculated = recalculation(doc_textblocks_json, doc_pagesizes)
            with open(os.path.join(outdir_recalculated_annotated_textblocks, filename.replace(".json", "_recalculated.json")), 'w', encoding='utf-8') as f:
                json.dump(doc_textblocks_recalculated, f, indent=2, ensure_ascii=False)
        except:
            pass

working on:  100084_Croll1609_Basilica_chymica_MDZ_MBS.json  ...
working on:  100094_Anon1625_Musaeum_hermeticum_VD17_SLUB.json  ...
working on:  100058_Hagecius1596_Actio_medica_ER_UBB.json  ...
working on:  100013_Ulstad1525_Coelum_philosophorum_Medica_BIUSP.json  ...
working on:  100069_Severinus1571_Idea_medicinae_philosophicae_GB_Noscemus.json  ...
working on:  100074_Hoghelande1595_De_lapidis_physici_conditionibus_MDZ_MBS.json  ...
working on:  100059_Suavius1567_Theophrasti_Paracelsi_Philosophiae_ONB.json  ...
working on:  100053_Mirandola1586_De_auro_libri_tres_MDZ_MBS.json  ...
working on:  100060_Witestein1583_Disceptatio_philosophica_MDZ_MBS.json  ...
working on:  100071_Pseudo-Lull1566_Testamentum_MDZ_MBS.json  ...
working on:  100085_Libavius1606_Commentariorum_alchemiae_pars_1_MDZ_MBS.json  ...
working on:  100062_Albertus1569_De_concordantia_Hippocraticorum_et_Paracelsistarum_MDZ_MBS.json  ...
working on:  100028_Pedemontanus1563_De_secretis_MDZ_MBS.json  ...
working on:

In [37]:
# first recalculate with raw annotated
for filename in textblocks_sanitized_filenames:
        with open(os.path.join(path_sanizited_textblocks, filename), 'r', ) as f:
            doc_textblocks_json = json.load(f)
        try:
            print("working on: ", filename, " ...")
            doc_pagesizes = docs_pagesizes[filename[:6]]
            doc_textblocks_recalculated = recalculation(doc_textblocks_json, doc_pagesizes)
            with open(os.path.join(outdir_recalculated_sanitized_textblocks, filename.replace(".json", "_recalculated.json")), 'w', encoding='utf-8') as f:
                json.dump(doc_textblocks_recalculated, f, indent=2, ensure_ascii=False)
        except:
            pass

working on:  100084_Croll1609_Basilica_chymica_MDZ_MBS.json  ...
working on:  100094_Anon1625_Musaeum_hermeticum_VD17_SLUB.json  ...
working on:  100058_Hagecius1596_Actio_medica_ER_UBB.json  ...
working on:  100013_Ulstad1525_Coelum_philosophorum_Medica_BIUSP.json  ...
working on:  100069_Severinus1571_Idea_medicinae_philosophicae_GB_Noscemus.json  ...
working on:  100074_Hoghelande1595_De_lapidis_physici_conditionibus_MDZ_MBS.json  ...
working on:  100059_Suavius1567_Theophrasti_Paracelsi_Philosophiae_ONB.json  ...
working on:  100053_Mirandola1586_De_auro_libri_tres_MDZ_MBS.json  ...
working on:  100060_Witestein1583_Disceptatio_philosophica_MDZ_MBS.json  ...
working on:  100071_Pseudo-Lull1566_Testamentum_MDZ_MBS.json  ...
working on:  100085_Libavius1606_Commentariorum_alchemiae_pars_1_MDZ_MBS.json  ...
working on:  100062_Albertus1569_De_concordantia_Hippocraticorum_et_Paracelsistarum_MDZ_MBS.json  ...
working on:  100028_Pedemontanus1563_De_secretis_MDZ_MBS.json  ...
working on: